1.Problema Monty Hall este un joc în care concurentul trebuie să aleagă una dintre cele trei uși, în spatele uneia aflându-se o mașină, iar în spatele celorlalte două, câte o capră. După ce concurentul alege o ușă, prezentatorul, care știe unde se află premiul, deschide o altă ușă în spatele căreia se află o capră. Concurentul poate apoi să își păstreze alegerea inițială sau să schimbe ușa, iar scopul este să afle care variantă îi oferă șanse mai mari de câștig. Pentru a rezolva problema, folosim biblioteca `pgmpy` din Python și construim o rețea Bayesiană formată din trei variabile: `C`, care reprezintă ușa aleasă de concurent, `P`, care indică ușa în spatele căreia se află premiul, și `H`, care reprezintă ușa deschisă de prezentator. Mai întâi, definim structura rețelei, în care variabilele `C` și `P` influențează variabila `H`, apoi stabilim distribuțiile de probabilitate pentru fiecare variabilă, presupunând că alegerea inițială și poziția premiului sunt aleatorii. Pentru variabila `H`, definim probabilitățile în funcție de ușa aleasă de concurent și de poziția premiului, ținând cont că prezentatorul nu poate deschide nici ușa concurentului, nici ușa în spatele căreia se află mașina. După adăugarea acestor distribuții în rețea, verificăm dacă modelul este corect și folosim algoritmul `VariableElimination` pentru a calcula probabilitatea poziției premiului, știind că participantul a ales ușa 0, iar prezentatorul a deschis ușa 2. Rezultatele arată că probabilitatea ca premiul să fie în spatele ușii 0 este de aproximativ 33,33%, în timp ce probabilitatea ca acesta să fie în spatele ușii 1 este de aproximativ 66,67%, iar ușa 2 are probabilitate zero, deoarece a fost deja deschisă și s-a demonstrat că ascunde o capră. Prin urmare, concluzia este că schimbarea ușii este strategia mai avantajoasă, deoarece dublează probabilitatea de câștig față de păstrarea alegerii inițiale.

Exercitiul 2

In [3]:

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD


model = DiscreteBayesianNetwork([
    ("B", "T"),
    ("B", "X"),
    ("B", "D"),
    ("T", "D")
])


cpd_b = TabularCPD(
    variable="B",
    variable_card=2,
    values=[[0.9], [0.1]]
)


cpd_t = TabularCPD(
    variable="T",
    variable_card=2,
    values=[
        [0.7, 0.2],  # T = 0
        [0.3, 0.8]   # T = 1
    ],
    evidence=["B"],
    evidence_card=[2]
)


cpd_x = TabularCPD(
    variable="X",
    variable_card=2,
    values=[
        [0.9, 0.1],  # X = 0
        [0.1, 0.9]   # X = 1
    ],
    evidence=["B"],
    evidence_card=[2]
)

cpd_d = TabularCPD(
    variable="D",
    variable_card=2,
    values=[
        [0.1, 0.6, 0.5, 0.9],  # D = 0
        [0.9, 0.4, 0.5, 0.1]   # D = 1
    ],
    evidence=["B", "T"],
    evidence_card=[2, 2]
)

model.add_cpds(cpd_b, cpd_t, cpd_x, cpd_d)

print("Model valid:", model.check_model())


Model valid: True


Exercitiul 3

In [4]:

import numpy as np

num_simulari = 10000
castiguri_J0 = 0
castiguri_J1 = 0

for _ in range(num_simulari):
    J0_incepe = np.random.choice([True, False])

    if J0_incepe:
        n = np.random.binomial(1, 0.5)
        m = np.random.binomial(2 * n, 4/7)

        if n >= m:
            castiguri_J0 += 1
        else:
            castiguri_J1 += 1
    else:
        n = np.random.binomial(1, 4/7)
        m = np.random.binomial(2 * n, 0.5)

        if n >= m:
            castiguri_J1 += 1
        else:
            castiguri_J0 += 1

print("J0:", castiguri_J0 / num_simulari)
print("J1:", castiguri_J1 / num_simulari)


J0: 0.488
J1: 0.512


In [5]:

import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD


model = DiscreteBayesianNetwork([
    ("S", "M"),
    ("N", "M")
])


cpd_s = TabularCPD(
    variable="S",
    variable_card=2,
    values=[[0.5], [0.5]]
)

cpd_n = TabularCPD(
    variable="N",
    variable_card=6,
    values=[[1/6] for _ in range(6)]
)



from math import comb

values_m = []

for m in range(13):
    row = []
    for s in range(2):
        for n in range(1, 7):
            p = 4/7 if s == 0 else 1/2

            if m <= 2*n:
                prob = comb(2*n, m) * p**m * (1-p)**(2*n-m)
            else:
                prob = 0

            row.append(prob)
    values_m.append(row)

cpd_m = TabularCPD(
    variable="M",
    variable_card=13,
    values=values_m,
    evidence=["S", "N"],
    evidence_card=[2, 6]
)

model.add_cpds(cpd_s, cpd_n, cpd_m)

print("Model valid:", model.check_model())


Model valid: True


In [6]:

from math import comb

def prob_o_stema(n, p):
    return comb(2*n, 1) * p * (1-p)**(2*n-1)

p_m1_J0 = sum(
    prob_o_stema(n, 4/7) / 6
    for n in range(1, 7)
)

p_m1_J1 = sum(
    prob_o_stema(n, 1/2) / 6
    for n in range(1, 7)
)

p_J0 = 0.5
p_J1 = 0.5

numitor = p_m1_J0 * p_J0 + p_m1_J1 * p_J1

posterior_J0 = p_m1_J0 * p_J0 / numitor
posterior_J1 = p_m1_J1 * p_J1 / numitor

print("P(J0 incepe | M=1):", posterior_J0)
print("P(J1 incepe | M=1):", posterior_J1)


P(J0 incepe | M=1): 0.4528939794932878
P(J1 incepe | M=1): 0.5471060205067123
